In [ ]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.llms import Ollama

from langchain.chains import RetrievalQA

from langchain.prompts import PromptTemplate

In [ ]:
loader = PyPDFLoader(r"E:\Downloads\LANGCHAIN\ebook.pdf")

documents = loader.load()

print("Pages loaded:", len(documents))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

final_document = text_splitter.split_documents(documents)

print("Chunks created:", len(final_document))

final_document[0]

In [ ]:
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device":"cuda"},
    encode_kwargs={"normalize_embeddings":True}
)

In [ ]:
vectorstore = FAISS.from_documents(
    final_document,
    huggingface_embeddings
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
prompt_template = """
You are an intelligent document question answering assistant.

Use ONLY the context provided below.

If the answer is not available in the context, reply:

"I couldn't find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context","question"]
)

In [ ]:
llm = Ollama(
    model="qwen2.5:7b",
    temperature=0
)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={
        "prompt": PROMPT
    },
    return_source_documents=True
)

In [ ]:
query = "What happened in 1991?"

result = qa_chain.invoke({"query":query})

print(result["result"])

In [ ]:
for doc in result["source_documents"]:
    print("="*80)
    print(doc.metadata)
    print(doc.page_content[:500])

In [ ]:
while True:

    query = input("\nAsk Question (type exit to quit): ")

    if query.lower() == "exit":
        break

    result = qa_chain.invoke({"query":query})

    print("\nAnswer:\n")
    print(result["result"])